In [1]:
!git clone https://github.com/4586245/DL_hw

Cloning into 'DL_hw'...
remote: Enumerating objects: 307, done.
remote: Counting objects: 100% (307/307), done.
remote: Compressing objects: 100% (233/233), done.
remote: Total 307 (delta 143), reused 159 (delta 62), pack-reused 0 (from 0)
Receiving objects: 100% (307/307), 73.26 KiB | 10.47 MiB/s, done.
Resolving deltas: 100% (143/143), done.


In [3]:
%cd DL_hw

/kaggle/working/DL_hw


In [4]:
!pip install -q hydra-core==1.3.4 comet_ml==3.58.4 soundfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.6/796.6 kB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 60.5 MB/s eta 0:00:00


In [5]:
import os

os.environ["ASVSPOOF_ROOT"] = (
    "/kaggle/input/datasets/awsaf49/asvpoof-2019-dataset/LA/LA"
)

In [6]:
from pathlib import Path

root = Path(os.environ["ASVSPOOF_ROOT"])

paths = [
    root / "ASVspoof2019_LA_train/flac",
    root / "ASVspoof2019_LA_dev/flac",
    root / "ASVspoof2019_LA_eval/flac",
    root / "ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt",
    root / "ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt",
    root / "ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt",
]

for path in paths:
    print(path.exists(), path)

True /kaggle/input/datasets/awsaf49/asvpoof-2019-dataset/LA/LA/ASVspoof2019_LA_train/flac
True /kaggle/input/datasets/awsaf49/asvpoof-2019-dataset/LA/LA/ASVspoof2019_LA_dev/flac
True /kaggle/input/datasets/awsaf49/asvpoof-2019-dataset/LA/LA/ASVspoof2019_LA_eval/flac
True /kaggle/input/datasets/awsaf49/asvpoof-2019-dataset/LA/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt
True /kaggle/input/datasets/awsaf49/asvpoof-2019-dataset/LA/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt
True /kaggle/input/datasets/awsaf49/asvpoof-2019-dataset/LA/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt


In [7]:
from kaggle_secrets import UserSecretsClient
import os

os.environ["COMET_API_KEY"] = UserSecretsClient().get_secret("COMET_API_KEY")

In [8]:
print("COMET_API_KEY loaded:", bool(os.environ.get("COMET_API_KEY")))

COMET_API_KEY loaded: True


In [9]:
!python preflight.py --tracker comet

tracker: Comet authentication passed (workspace=pn-grebennikova)
train: {'total': 25380, 'bonafide': 2580, 'spoof': 22800}
dev: {'total': 24844, 'bonafide': 2548, 'spoof': 22296}
device: Tesla T4
features: (16, 64, 600)
logits: (16, 2)
loss: 0.703140
PREFLIGHT PASSED: the full highscore run can be started.


In [ ]:
!python train.py -cn=highscore_comet

In [ ]:
!cp saved/lcnn-mel-highscore/model_best.pth /kaggle/working/model_best.pth

In [ ]:
LOGIN = "ponigrebennikova"

In [ ]:
!python inference.py -cn=inference_mel \
  inferencer.from_pretrained=/kaggle/working/model_best.pth \
  inferencer.output_filename={LOGIN}.csv

In [ ]:
protocol = (
    f"{os.environ['ASVSPOOF_ROOT']}/"
    "ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt"
)
submission = f"data/saved/asvspoof_mel/{LOGIN}.csv"

!python validate_submission.py "{submission}" "{protocol}"

In [ ]:
from pathlib import Path
import shutil

repo = Path.cwd()
grading_dir = Path("/kaggle/working/grading_check")
solutions_dir = grading_dir / "students_solutions"
solutions_dir.mkdir(parents=True, exist_ok=True)

shutil.copy(repo / "grading.py", grading_dir / "grading.py")
shutil.copy(repo / "calculate_eer.py", grading_dir / "calculate_eer.py")
shutil.copy(protocol, grading_dir / "ASVspoof2019.LA.cm.eval.trl.txt")
shutil.copy(submission, solutions_dir / f"{LOGIN}.csv")

In [ ]:
%cd /kaggle/working/grading_check
!python grading.py
!cat grades.csv

In [ ]:
!cp "students_solutions/{LOGIN}.csv" "/kaggle/working/{LOGIN}.csv"